# 概率与分布（Probability and Distributions）

对应课程：`phases/01-math-foundations/06-probability-and-distributions`

> 概率是 AI 用来表达不确定性的语言。

本 notebook 把 `probability.py` 里的核心函数拆开：每个函数一组中文注释，后面跟一小段可运行实验。完整打印型 demo 仍在 `probability.py`。

**贯穿全课的模式：** 分布给出 $P(x)$；期望/方差是分布的摘要；softmax 把 logits 变成分布；交叉熵衡量两个分布有多不像。


## 学习目标（Learning Objectives）

- 从零实现 Bernoulli、categorical、Poisson、均匀、正态的 PMF/PDF
- 算期望、方差，并用中心极限定理解释高斯为何到处出现
- 用减 max 做数值稳定的 softmax / log-softmax
- 从 logits 算交叉熵，并连到负对数似然


## 0. 依赖

只用标准库。采样实验固定种子，结果可复现。


In [1]:
import math
import random

random.seed(42)


## 1. 组合数 $C(n,k)$

$$
C(n,k)=\frac{n!}{k!(n-k)!}
$$

用整数除法，避免先算出巨大浮点阶乘再除。


In [2]:
def factorial(n):
    """n!，从 2 累乘到 n。"""
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result


def combinations(n, k):
    """C(n,k) = n! / (k! (n-k)!)，整数除法。"""
    return factorial(n) // (factorial(k) * factorial(n - k))


print("C(5,2) =", combinations(5, 2), "  (期望 10)")


C(5,2) = 10   (期望 10)


## 2. 条件概率

$$
P(A\mid B)=\frac{P(A\cap B)}{P(B)}
$$

扑克：52 张里 4 张 K，12 张人头牌。$P(\text{K}\mid\text{人头})=4/12$。


In [3]:
def conditional_probability(p_a_and_b, p_b):
    """P(A|B) = P(A and B) / P(B)。调用方保证 p_b > 0。"""
    return p_a_and_b / p_b


p = conditional_probability(4 / 52, 12 / 52)
print("P(King | Face card) =", round(p, 4), "  (期望 0.3333)")


P(King | Face card) = 0.3333   (期望 0.3333)


## 3. 离散分布：Bernoulli / Categorical / Poisson

- Bernoulli：$P(X=1)=p$，$P(X=0)=1-p$
- Categorical：有限个类别，概率表直接查
- Poisson：$P(X=k)=\lambda^k e^{-\lambda}/k!$


In [4]:
def bernoulli_pmf(k, p):
    """两点分布：k 只能是 0 或 1。"""
    return p if k == 1 else (1 - p)


def categorical_pmf(k, probs):
    """类别 k 的概率就是表里第 k 项。"""
    return probs[k]


def poisson_pmf(k, lam):
    """计数分布：均值 = 方差 = lambda。"""
    return (lam ** k) * math.exp(-lam) / factorial(k)


print("Bernoulli p=0.7: P(0) =", bernoulli_pmf(0, 0.7), "P(1) =", bernoulli_pmf(1, 0.7))
print("Categorical [0.1,0.3,0.6] P(1) =", categorical_pmf(1, [0.1, 0.3, 0.6]))
print("Poisson λ=3  P(0..4) =", [round(poisson_pmf(k, 3), 4) for k in range(5)])


Bernoulli p=0.7: P(0) = 0.30000000000000004 P(1) = 0.7
Categorical [0.1,0.3,0.6] P(1) = 0.3
Poisson λ=3  P(0..4) = [0.0498, 0.1494, 0.224, 0.224, 0.168]


## 4. 连续密度：均匀与正态

$$
U(x;a,b)=\frac{1}{b-a}\mathbf{1}_{[a,b]}(x)
$$

$$
\mathcal{N}(x;\mu,\sigma)=\frac{1}{\sigma\sqrt{2\pi}}\exp\left(-\frac12\left(\frac{x-\mu}{\sigma}\right)^2\right)
$$

PDF 在一点的值不是概率；正态在均值处最高。


In [5]:
def uniform_pdf(x, a, b):
    """区间 [a,b] 上高度恒为 1/(b-a)，外面为 0。"""
    if a <= x <= b:
        return 1.0 / (b - a)
    return 0.0


def normal_pdf(x, mu, sigma):
    """高斯密度。coeff 是归一化常数，exponent 是马氏距离的一半。"""
    coeff = 1.0 / (sigma * math.sqrt(2 * math.pi))
    exponent = -0.5 * ((x - mu) / sigma) ** 2
    return coeff * math.exp(exponent)


print("U[0,2] at 1 =", uniform_pdf(1.0, 0.0, 2.0), "  (期望 0.5)")
print("U[0,2] at 3 =", uniform_pdf(3.0, 0.0, 2.0), "  (期望 0)")
peak = normal_pdf(0.0, 0.0, 1.0)
print("N(0,1) at mean =", round(peak, 6), "  (期望 1/sqrt(2π) ≈", round(1 / math.sqrt(2 * math.pi), 6), ")")


U[0,2] at 1 = 0.5   (期望 0.5)
U[0,2] at 3 = 0.0   (期望 0)
N(0,1) at mean = 0.398942   (期望 1/sqrt(2π) ≈ 0.398942 )


## 5. 期望与方差

$$
\mathbb{E}[X]=\sum_i x_i p_i,\qquad
\mathrm{Var}(X)=\sum_i p_i(x_i-\mu)^2
$$

公平硬币：$X\in\{0,1\}$，各 $1/2$。$\mathbb{E}=1/2$，$\mathrm{Var}=1/4$。


In [6]:
def expected_value(values, probabilities):
    """对取值按概率加权求和。"""
    return sum(v * p for v, p in zip(values, probabilities))


def variance(values, probabilities):
    """先算均值，再算平方偏差的期望。"""
    mu = expected_value(values, probabilities)
    return sum(p * (v - mu) ** 2 for v, p in zip(values, probabilities))


coin_vals = [0, 1]
coin_ps = [0.5, 0.5]
print("fair coin E[X] =", expected_value(coin_vals, coin_ps))
print("fair coin Var  =", variance(coin_vals, coin_ps))


fair coin E[X] = 0.5
fair coin Var  = 0.25


## 6. 采样：Bernoulli 与 Box–Muller 正态

Bernoulli：抽 $U\sim\mathrm{Unif}(0,1)$，小于 $p$ 则记 1。

Box–Muller：两个均匀数变成一个标准正态，再仿射到 $\mathcal{N}(\mu,\sigma^2)$。


In [7]:
def sample_bernoulli(p, n=1):
    """独立掷 n 次偏置硬币。"""
    return [1 if random.random() < p else 0 for _ in range(n)]


def sample_normal_box_muller(mu, sigma, n=1):
    """u1,u2 ~ U(0,1) → z = sqrt(-2 ln u1) cos(2π u2) ~ N(0,1)。"""
    samples = []
    for _ in range(n):
        u1 = random.random()
        u2 = random.random()
        z = math.sqrt(-2 * math.log(u1)) * math.cos(2 * math.pi * u2)
        samples.append(mu + sigma * z)
    return samples


bern = sample_bernoulli(0.3, 20)
print("Bernoulli(0.3) x20:", bern, "  mean", round(sum(bern) / len(bern), 3))
norm = sample_normal_box_muller(0.0, 1.0, 200)
m = sum(norm) / len(norm)
v = sum((x - m) ** 2 for x in norm) / len(norm)
print("N(0,1) x200: mean", round(m, 3), "var", round(v, 3))


Bernoulli(0.3) x20: [0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1]   mean 0.5
N(0,1) x200: mean 0.003 var 0.949


## 7. Softmax、log-softmax、交叉熵

$$
\mathrm{softmax}(z_i)=\frac{e^{z_i}}{\sum_j e^{z_j}}
=\frac{e^{z_i-c}}{\sum_j e^{z_j-c}},\quad c=\max z
$$

$$
\log\mathrm{softmax}(z_i)=z_i-\mathrm{LSE}(z),\qquad
\mathcal{L}=-\log p_y
$$

减 max 是为了 `exp` 不炸；交叉熵走对数空间，避免 `log(0)`。


In [8]:
def softmax(logits):
    """减 max 再 exp，再归一化成概率。"""
    max_logit = max(logits)
    shifted = [z - max_logit for z in logits]
    exps = [math.exp(z) for z in shifted]
    total = sum(exps)
    return [e / total for e in exps]


def log_softmax(logits):
    """z_i - logsumexp(z)，全程对数空间。"""
    max_logit = max(logits)
    shifted = [z - max_logit for z in logits]
    log_sum_exp = max_logit + math.log(sum(math.exp(z) for z in shifted))
    return [z - log_sum_exp for z in logits]


def cross_entropy_loss(logits, target_index):
    """分类损失 = -log p_target。"""
    log_probs = log_softmax(logits)
    return -log_probs[target_index]


logits = [1.0, 2.0, 3.0]
probs = softmax(logits)
print("softmax([1,2,3]) =", [round(p, 4) for p in probs])
print("sum =", round(sum(probs), 6))
print("log_softmax =", [round(v, 4) for v in log_softmax(logits)])
print("CE target=2 =", round(cross_entropy_loss(logits, 2), 4))


softmax([1,2,3]) = [0.09, 0.2447, 0.6652]
sum = 1.0
log_softmax = [-2.4076, -1.4076, -0.4076]
CE target=2 = 0.4076


## 8. 联合 → 边缘，以及独立性

$$
P(X=i)=\sum_j P(X=i,Y=j),\qquad
\text{独立} \iff P(X,Y)=P(X)P(Y)
$$


In [9]:
def joint_to_marginals(joint):
    """行求和得 P(X)，列求和得 P(Y)。"""
    rows = len(joint)
    cols = len(joint[0])
    marginal_x = [sum(joint[i][j] for j in range(cols)) for i in range(rows)]
    marginal_y = [sum(joint[i][j] for i in range(rows)) for j in range(cols)]
    return marginal_x, marginal_y


def check_independence(joint, marginal_x, marginal_y, tol=1e-9):
    """逐格比较 joint[i][j] 与 P(X=i)P(Y=j)。"""
    for i in range(len(marginal_x)):
        for j in range(len(marginal_y)):
            if abs(joint[i][j] - marginal_x[i] * marginal_y[j]) > tol:
                return False
    return True


dependent = [
    [0.40, 0.10],
    [0.05, 0.45],
]
mx, my = joint_to_marginals(dependent)
print("dependent marginal X,Y =", mx, my)
print("independent?", check_independence(dependent, mx, my))

independent = [
    [0.25, 0.25],
    [0.25, 0.25],
]
mx, my = joint_to_marginals(independent)
print("independent table? ", check_independence(independent, mx, my))


dependent marginal X,Y = [0.5, 0.5] [0.45, 0.55]
independent? False
independent table?  True


## 9. 中心极限定理（小样本）

独立同分布变量的样本均值，样本量变大时逼近正态。下面：每次掷 30 枚 $p=0.5$ 的硬币取平均，重复 200 次。均值应靠近 0.5，标准差应靠近 $\sqrt{p(1-p)/n}=1/\sqrt{120}\approx 0.091$。


In [10]:
def demonstrate_clt(dist_fn, n_per_sample, n_averages):
    """对 dist_fn 抽 n_per_sample 个，求平均；重复 n_averages 次。"""
    averages = []
    for _ in range(n_averages):
        samples = [dist_fn() for _ in range(n_per_sample)]
        averages.append(sum(samples) / len(samples))
    return averages


random.seed(42)
avgs = demonstrate_clt(lambda: 1 if random.random() < 0.5 else 0, 30, 200)
mean = sum(avgs) / len(avgs)
std = (sum((x - mean) ** 2 for x in avgs) / len(avgs)) ** 0.5
print("200 averages of 30 Bernoulli(0.5)")
print("mean =", round(mean, 4), "  (期望 ~0.5)")
print("std  =", round(std, 4), "  (期望 ~", round((0.25 / 30) ** 0.5, 4), ")")
print("min/max =", round(min(avgs), 3), round(max(avgs), 3))


200 averages of 30 Bernoulli(0.5)
mean = 0.4955   (期望 ~0.5)
std  = 0.0855   (期望 ~ 0.0913 )
min/max = 0.267 0.8


## 对照表

| 函数 | 角色 |
|------|------|
| `combinations` | 组合计数 $C(n,k)$ |
| `conditional_probability` | $P(A\mid B)=P(A,B)/P(B)$ |
| `bernoulli_pmf` / `categorical_pmf` / `poisson_pmf` | 离散 PMF |
| `uniform_pdf` / `normal_pdf` | 连续密度 |
| `expected_value` / `variance` | 一阶、二阶矩 |
| `sample_bernoulli` / `sample_normal_box_muller` | 从分布抽样本 |
| `softmax` / `log_softmax` / `cross_entropy_loss` | logits → 概率 → 分类损失 |
| `joint_to_marginals` / `check_independence` | 联合表的边缘与独立性 |
| `demonstrate_clt` | 样本均值趋向正态 |

要看完整打印 demo，运行：

```bash
python probability.py
```
